# 3. Same complex, small sequence changes

Structures of one complex that differ by a few residues show what a mutation changes. The
A6 TCR on HLA-A*02:01 has been solved with the Tax peptide `LLFGYPVYV` and with
variants of it (Ding et al., *Immunity* 1999), and as a high-affinity TCR with a mutated
CDR3β (4FTV):

| PDB | change |
|---|---|
| 1AO7 | wild type |
| 1QRN | peptide P6A |
| 1QSE | peptide V7R |
| 1QSF | peptide Y8A |
| 4FTV | CDR3β `LAGG` to `MSAQ` (high-affinity A6) |

Each variant is scored against 1AO7 as if it were a model. The structures are downloaded
from the RCSB.

In [1]:
import pandas as pd
from pdbframe import SeqFrame

import tcrstruct as tcs
from tcrstruct import TCRFrame

pd.options.display.precision = 3

In [2]:
ids = ['1AO7', '1QRN', '1QSE', '1QSF', '4FTV']
structures = {i: TCRFrame.from_rcsb(i) for i in ids}
wild_type = structures['1AO7']
tcs.sources.rcsb_info(ids)[['pdb_id', 'title', 'resolution']]

,pdb_id,title,resolution
0,1AO7,"COMPLEX BETWEEN HUMAN T-CELL RECEPTOR, VIRAL P...",2.60
1,1QRN,CRYSTAL STRUCTURE OF HUMAN A6 TCR COMPLEXED WI...,2.80
2,1QSE,STRUCTURE OF HUMAN A6-TCR BOUND TO HLA-A2 COMP...,2.80
3,1QSF,STRUCTURE OF A6-TCR BOUND TO HLA-A2 COMPLEXED ...,2.80
4,4FTV,The complex between the high affinity version ...,2.74


## What differs in sequence

The peptides, and the CDR3β loops, aligned to the wild type (`.` is the same residue):

In [3]:
SeqFrame.align(*[s.peptide for s in structures.values()], names=ids).show(reference=True)

1AO7     1 LLFGYPVYV 9
1QRN     1 .....A... 9
1QSE     1 ......R.. 9
1QSF     1 .......A. 9
4FTV     1 ......... 9
           |||||...|



In [4]:
SeqFrame.align(*[s.cdrs['CDRB3'] for s in structures.values()], names=ids).show(reference=True)

1AO7     1 ASRPGLAGGRPEQY 14
1QRN     1 .............. 14
1QSE     1 .............. 14
1QSF     1 .............. 14
4FTV     1 ......MSAQ.... 14
           ||||||..:.||||



## What differs in structure

Residues are matched only where the amino acid is the same, so **a mutated position is
left out** of the RMSDs and contacts, and everything else is scored as usual.
`Peptide` is over 8 residues for the peptide variants, and 4FTV's `CDRB3` does not
include its four mutated residues: it measures the loop around them.

In [5]:
variants = {i: structures[i] for i in ids[1:]}
scores = tcs.compare_models(variants, wild_type, grouped=True)
scores[['peptide', 'cdrs']]

group  peptide                                    cdrs                       \
column Peptide Peptide_Centric peptide_reversed  CDRA1  CDRA2  CDRA3  CDRB1   
model                                                                         
1QRN     0.271           1.030            False  0.463  0.498  0.668  0.434   
1QSE     0.577           1.181            False  0.730  0.846  0.520  0.818   
1QSF     0.424           1.016            False  0.852  0.410  0.557  0.887   
4FTV     0.635           1.131            False  2.211  1.136  0.637  0.842   

group                          
column  CDRB2  CDRB3 Mean_CDR  
model                          
1QRN    0.346  1.685    0.682  
1QSE    0.791  2.308    1.002  
1QSF    0.722  1.192    0.770  
4FTV    1.043  0.669    1.090

In [6]:
scores[['global', 'docking', 'angles']]

group  global                 docking                                        \
column   Full   lddt tm_score   DockQ  iRMSD  LRMSD   fnat     F1 TCR_LRMSD   
model                                                                         
1QRN    1.050  0.933    0.996   0.909  0.650  0.856  0.896  0.916     0.845   
1QSE    1.062  0.938    0.995   0.850  0.834  1.101  0.803  0.862     0.882   
1QSF    1.276  0.925    0.994   0.859  0.759  1.405  0.806  0.864     1.320   
4FTV    1.411  0.898    0.994   0.926  0.655  1.354  0.962  0.927     0.992   

group                     angles                                      \
column pMHC_LRMSD crossing_angle incident_angle crossing_angle_error   
model                                                                  
1QRN        0.424         32.448         11.353                1.083   
1QSE        0.700         35.796         12.514                2.265   
1QSF        0.583         34.092         13.338                0.561   
4FTV        1.027         32.903         12.310                0.627   

group                        
column incident_angle_error  
model                        
1QRN                  0.046  
1QSE                  1.207  
1QSF                  2.031  
4FTV                  1.048

The docking angles are nearly the same in every structure: the TCR docks the same
way.

In [7]:
pd.DataFrame({i: s.angles for i, s in structures.items()})

,1AO7,1QRN,1QSE,1QSF,4FTV
crossing_angle,33.531,32.448,35.796,34.092,32.903
incident_angle,11.307,11.353,12.514,13.338,12.331


The variants on the wild type, fitted on the pMHC and colored by per-residue lDDT.
Hover a residue to see its partners in the other structures.

In [8]:
tcs.view_comparison(variants, wild_type, superpose=['A', 'C'],
                    regions=['peptide_core', 'cdr3'])

## Where a variant moves

1QSE (V7R) has the largest CDR3β RMSD. The per-residue lDDT of the peptide and CDR3β
residues shows which ones moved (position 7, mutated, is not there):

In [9]:
per_residue = tcs.per_residue_lddt(structures['1QSE'], wild_type)
per_residue.query("region in ['peptide_core', 'CDRB3']")

,chain,residue_number,insertion_code,aa,imgt,region,lddt
274,C,1,,L,-,peptide_core,0.915
275,C,2,,L,-,peptide_core,0.974
276,C,3,,F,-,peptide_core,0.938
277,C,4,,G,-,peptide_core,0.942
278,C,5,,Y,-,peptide_core,0.873
279,C,6,,P,-,peptide_core,0.905
280,C,8,,Y,-,peptide_core,0.948
281,C,9,,V,-,peptide_core,0.973
486,E,93,,A,105,CDRB3,0.970
487,E,94,,S,106,CDRB3,0.929
